# A1.4 - Sweep Parametrów Środowiska
## Robot Recyklingowy: Kiedy zmienia się optymalna polityka?

Zadanie: zrozumieć, że parametry środowiska wpływają na to jaka polityka jest optymalna.
- A1.2: porównujemy 3 polityki
- A1.3: znajdujemy najlepszą politykę
- A1.4: zmieniamy parametry i patrzymy co się zmienia

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Funkcja do rozwiązywania równania Bellmana
def evaluate_policy_linear_system(P_pi: np.ndarray, r_pi: np.ndarray, gamma: float) -> np.ndarray:
    """Liczy V_pi rozwiązując (I - gamma*P_pi)*v = r_pi"""
    nS = P_pi.shape[0]
    I = np.eye(nS)
    return np.linalg.solve(I - gamma * P_pi, r_pi)

# Budowanie modelu robota
def build_recycling_robot_P(alpha=0.8, beta=0.4, r_search=5.0, r_wait=1.0, rescue_cost=-3.0):
    """MDP robota recyklingowego
    Stany: H=0 (wysoka energia), L=1 (niska energia)
    Akcje: SEARCH=0, WAIT=1, RECHARGE=2
    """
    nS, nA = 2, 3
    P = {s: {a: [] for a in range(nA)} for s in range(nS)}
    
    H, L = 0, 1
    SEARCH, WAIT, RECHARGE = 0, 1, 2
    
    # Stan H
    P[H][SEARCH] = [(alpha, H, r_search, False), (1 - alpha, L, r_search, False)]
    P[H][WAIT] = [(1.0, H, r_wait, False)]
    P[H][RECHARGE] = []  # niedostępna
    
    # Stan L
    P[L][SEARCH] = [(beta, L, r_search, False), (1 - beta, H, rescue_cost, False)]
    P[L][WAIT] = [(1.0, L, r_wait, False)]
    P[L][RECHARGE] = [(1.0, H, 0.0, False)]
    
    return P, nS, nA

# Budowanie P_pi i r_pi dla polityki
def build_P_r_for_policy(P, pi):
    nS = len(P)
    nA = len(P[0])
    P_pi = np.zeros((nS, nS), dtype=float)
    r_pi = np.zeros(nS, dtype=float)
    
    for s in range(nS):
        for a in range(nA):
            w = float(pi[s, a])
            if w == 0.0:
                continue
            outcomes = P[s][a]
            if not outcomes:
                continue
            for (p, s2, r, terminated) in outcomes:
                r_pi[s] += w * p * float(r)
                if not terminated:
                    P_pi[s, int(s2)] += w * p
    return P_pi, r_pi

print("Setup OK")

## Co robimy?

Mamy robota z baterią w 2 stanach: wysoka energia (H) i niska energia (L).

Może on:
- SEARCH: szuka puszek (+5), ale może się rozładować
- WAIT: czeka na człowieka (+1), bezpieczne
- RECHARGE: ładuje baterię (tylko w L, wraca do H)

Pytanie: Jaka polityka (strategia) jest najlepsza?

Ważne:
- Parametry środowiska (alpha, beta, rescue_cost) to nie jest polityka
- To definiuje jak świat działa
- Polityka to co robot robi (którą akcję wybiera)
- V_pi(s) to ile średnio zarobi jeśli będzie stosować politykę pi

## A1.2 - Porównanie polityk

In [ ]:
gamma = 0.9
P, nS, nA = build_recycling_robot_P()

H, L = 0, 1
SEARCH, WAIT, RECHARGE = 0, 1, 2

print("Testujemy 3 różne polityki w tym samym środowisku\n")

# Polityka 1: intuicyjna
print("1. Polityka 1: SEARCH w H, RECHARGE w L")
print("   (logika: szukamy gdy mamy energię, ładujemy gdy brakuje)\n")
pi1 = np.zeros((nS, nA))
pi1[H, SEARCH] = 1.0
pi1[L, RECHARGE] = 1.0

P_pi1, r_pi1 = build_P_r_for_policy(P, pi1)
v1 = evaluate_policy_linear_system(P_pi1, r_pi1, gamma)

print(f"   V(H) = {v1[0]:.2f}")
print(f"   V(L) = {v1[1]:.2f}")
print(f"   Średnia: {np.mean(v1):.2f}\n")

# Polityka 2: konserwatywna
print("2. Polityka 2: WAIT wszędzie")
print("   (logika: nigdy nie szukamy, tylko czekamy)\n")
pi2 = np.zeros((nS, nA))
pi2[H, WAIT] = 1.0
pi2[L, WAIT] = 1.0

P_pi2, r_pi2 = build_P_r_for_policy(P, pi2)
v2 = evaluate_policy_linear_system(P_pi2, r_pi2, gamma)

print(f"   V(H) = {v2[0]:.2f}")
print(f"   V(L) = {v2[1]:.2f}")
print(f"   Średnia: {np.mean(v2):.2f}\n")

# Polityka 3: agresywna
print("3. Polityka 3: SEARCH wszędzie")
print("   (logika: zawsze szukamy, bez ostrożności)\n")
pi3 = np.zeros((nS, nA))
pi3[H, SEARCH] = 1.0
pi3[L, SEARCH] = 1.0

P_pi3, r_pi3 = build_P_r_for_policy(P, pi3)
v3 = evaluate_policy_linear_system(P_pi3, r_pi3, gamma)

print(f"   V(H) = {v3[0]:.2f}")
print(f"   V(L) = {v3[1]:.2f}")
print(f"   Średnia: {np.mean(v3):.2f}\n")

print("="*50)
results = pd.DataFrame({
    'Polityka': ['SEARCH-H, RECHARGE-L', 'WAIT wszędzie', 'SEARCH wszędzie'],
    'V(H)': [f"{v1[0]:.2f}", f"{v2[0]:.2f}", f"{v3[0]:.2f}"],
    'V(L)': [f"{v1[1]:.2f}", f"{v2[1]:.2f}", f"{v3[1]:.2f}"],
    'Średnia': [f"{np.mean(v1):.2f}", f"{np.mean(v2):.2f}", f"{np.mean(v3):.2f}"]
})
print(results.to_string(index=False))
print("="*50)
print(f"\nNajlepsza: polityka 1 (średnia {np.mean(v1):.2f})")
print("Wnioski: balans między ryzykiem a zyskiem daje najlepszy wynik")

## A1.3 - Znalezienie optymalnej polityki

In [ ]:
def all_deterministic_policies_robot():
    """Generuje wszystkie 6 możliwych polityk"""
    H, L = 0, 1
    SEARCH, WAIT, RECHARGE = 0, 1, 2
    policies = []
    for aH in [SEARCH, WAIT]:
        for aL in [SEARCH, WAIT, RECHARGE]:
            pi = np.zeros((2, 3))
            pi[H, aH] = 1.0
            pi[L, aL] = 1.0
            policies.append(pi)
    return policies

def best_policy_robot(P, gamma=0.9):
    """Sprawdza wszystkie polityki i zwraca najlepszą"""
    best_pi = None
    best_vH = None
    for pi in all_deterministic_policies_robot():
        P_pi, r_pi = build_P_r_for_policy(P, pi)
        v = evaluate_policy_linear_system(P_pi, r_pi, gamma)
        vH = float(v[0])
        if best_vH is None or vH > best_vH:
            best_vH = vH
            best_pi = pi
    return best_pi, best_vH

print("Znajdujemy najlepszą politykę")
print("Robot ma tylko 2 stany i kilka akcji,")
print("więc możemy sprawdzić wszystkie 6 możliwych polityk.\n")

P, nS, nA = build_recycling_robot_P()
action_name = {0: "SEARCH", 1: "WAIT", 2: "RECHARGE"}

pi_star, vH_star = best_policy_robot(P, gamma=0.9)
P_pi_star, r_pi_star = build_P_r_for_policy(P, pi_star)
v_star = evaluate_policy_linear_system(P_pi_star, r_pi_star, 0.9)

aH = action_name[int(np.argmax(pi_star[0]))]
aL = action_name[int(np.argmax(pi_star[1]))]

print(f"Parametry: alpha=0.8, beta=0.4, r_search=5, r_wait=1, rescue_cost=-3, gamma=0.9\n")
print(f"Optymalna polityka:")
print(f"  W H (wysoka energia): {aH}")
print(f"  W L (niska energia): {aL}\n")
print(f"Wartości:")
print(f"  V*(H) = {v_star[0]:.2f}")
print(f"  V*(L) = {v_star[1]:.2f}")
print(f"\nTo jest dokładnie ta sama polityka co polityka 1!")
print("Czyli nasza intuicja była prawidłowa.")
print("\nAle czy zawsze będzie najlepsza?")

## A1.4 - Sweep parametrów

In [ ]:
print("Teraz zmieniamy parametry i patrzymy jak zmienia się optymalna polityka\n")

alpha = 0.8
r_search = 5.0
r_wait = 1.0
gamma = 0.9

beta_list = [0.1, 0.3, 0.5, 0.7, 0.9]
rescue_list = [-1.0, -3.0, -6.0, -10.0]
action_name = {0: "SEARCH", 1: "WAIT", 2: "RECHARGE"}

print(f"Parametry stałe: alpha={alpha}, r_search={r_search}, r_wait={r_wait}, gamma={gamma}")
print(f"Parametry zmienne: beta={beta_list}")
print(f"                   rescue_cost={rescue_list}\n")

print(f"{'beta':<6} {'rescue':<10} {'pi*(H)':<12} {'pi*(L)':<12}")
print("-" * 42)

for beta in beta_list:
    for rescue_cost in rescue_list:
        P, _, _ = build_recycling_robot_P(alpha=alpha, beta=beta,
                                          r_search=r_search, r_wait=r_wait,
                                          rescue_cost=rescue_cost)
        pi_star, _ = best_policy_robot(P, gamma=gamma)
        aH = action_name[int(np.argmax(pi_star[0]))]
        aL = action_name[int(np.argmax(pi_star[1]))]
        print(f"{beta:<6.1f} {rescue_cost:<10.0f} {aH:<12} {aL:<12}")

print("\n" + "="*50)
print("Co widzimy:")
print("="*50)
print(f"""
1. pi*(H) zawsze SEARCH
   Bo w stanie H mamy energię, więc SEARCH jest bezpieczny.
   Dlaczego czekać na +1 jeśli możemy dostawać +5?

2. pi*(L) zmienia się!
   
   Gdy beta rośnie (SEARCH w L jest bezpieczniejszy):
   - beta=0.1: zawsze RECHARGE (zbyt ryzykowne SEARCH)
   - beta=0.5: zmienia się zależnie od rescue_cost
   - beta=0.9: zawsze SEARCH (prawie zawsze się uda)
   
   Gdy rescue_cost maleje (kara mniejsza):
   - rescue=-1.0: większa skłonność do SEARCH
   - rescue=-10.0: większa skłonność do RECHARGE

3. Istnieje PRÓG (dla beta około 0.5-0.7)
   Poniżej niego pi*(L) = RECHARGE
   Powyżej niego pi*(L) = SEARCH
   
4. Ale dokładna wartość progu zależy od rescue_cost

WNIOSEK:
Parametry środowiska (jak ryzykowny jest SEARCH, jak duża kara za awarię)
wpływają na to jaka polityka jest optymalna.
Robot musi się dostosowywać!
""")

## Drobna analiza

In [ ]:
# Przyjrzyjmy się konkretnym przejściom
print("Gdzie dokładnie zmienia się pi*(L)?\n")

for beta in [0.5, 0.7]:
    print(f"beta = {beta}:")
    for rescue_cost in rescue_list:
        P, _, _ = build_recycling_robot_P(alpha=alpha, beta=beta,
                                          r_search=r_search, r_wait=r_wait,
                                          rescue_cost=rescue_cost)
        pi_star, _ = best_policy_robot(P, gamma=gamma)
        aL = action_name[int(np.argmax(pi_star[1]))]
        print(f"  rescue_cost={rescue_cost:6.1f} -> pi*(L) = {aL}")
    print()

print("\nWidać, że dla beta=0.5 i beta=0.7 jest przejście.")
print("Gdy rescue_cost=-1.0 (mała kara), SEARCH się opłaca.")
print("Gdy rescue_cost=-3.0 lub mniejsze, RECHARGE jest bezpieczniejszy.")
print("\nTo ma ekonomiczny sens - im wyższa kara za porażkę, tym ostrożniej.")

## Podsumowanie

### Co się stało:

**A1.2** - Porównaliśmy 3 polityki w tym samym środowisku.
Polityka intuicyjna (szukamy gdy energia wysoka, ładujemy gdy niska) okazała się najlepsza.

**A1.3** - Znaleźliśmy optymalną politykę sprawdzając wszystkie 6 możliwości.
Okazała się być tą samą polityką co intuicyjna.

**A1.4** - Zmieniliśmy parametry środowiska i obserwowaliśmy co się zmienia.
Okazało się, że gdy zmienia się ryzyko (beta) lub kara (rescue_cost),
zmienia się też optymalna polityka!

### Kluczowe spostrzeżenia:

1. **Polityka to nie to samo co parametry środowiska**
   - Parametry (alpha, beta, rescue_cost) opisują świat
   - Polityka (którą akcję wybrać) opisuje zachowanie agenta
   - To są różne rzeczy!

2. **V_pi(s) mierzy jakość polityki**
   - Dla danej polityki i danego środowiska
   - Im wyższa wartość, tym lepsza polityka

3. **Parametry wpływają na optymalną politykę**
   - Gdy świat się zmienia, optymalna strategia też się zmienia
   - To nie jest bug - to naturalne!

4. **Możemy znaleźć pi* na różne sposoby**
   - Brute force: sprawdzić wszystkie (robimy teraz)
   - DP algorithms: value iteration, policy iteration (następnie rozdział)